# Robot URDF Visualization and Manipulation with Drake and MeshCat

This notebook demonstrates how to load, visualize, and interactively manipulate a robot URDF file using Drake simulator and MeshCat visualization.

## 1. Import Required Libraries

Import necessary libraries including Drake, MeshCat, NumPy, and other dependencies required for visualization and manipulation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import ipywidgets as widgets
from ipywidgets import FloatSlider, HBox, VBox, interactive_output

# Drake imports
from pydrake.systems.framework import DiagramBuilder
from pydrake.systems.primitives import ConstantVectorSource, Multiplexer
from pydrake.multibody.parsing import Parser
from pydrake.multibody.plant import MultibodyPlant, AddMultibodyPlantSceneGraph
from pydrake.geometry import StartMeshcat
from pydrake.systems.meshcat_visualizer import MeshcatVisualizer, MeshcatVisualizerParams
from pydrake.common.value import Value
from pydrake.systems.analysis import Simulator

print("All libraries imported successfully!")

## 2. Load URDF File into Drake

Load the robot URDF file using Drake's MultibodyPlant and SceneGraph to create a dynamic model of the robot.

In [ ]:
# Create a MultibodyPlant and SceneGraph
plant = MultibodyPlant(time_step=0.0)
scene_graph = plant.scene_graph()

# Load the robot URDF file
parser = Parser(plant)
robot_urdf_path = "model/robot.urdf"

try:
    robot_model = parser.AddModelFromFile(robot_urdf_path)
    print(f"Successfully loaded robot from: {robot_urdf_path}")
except Exception as e:
    print(f"Error loading URDF: {e}")
    print(f"Make sure the robot.urdf file exists at: {robot_urdf_path}")

# Finalize the plant
plant.Finalize()

# Get plant information
num_q = plant.num_positions()
num_v = plant.num_velocities()

print(f"\nRobot Configuration:")
print(f"  Number of generalized positions: {num_q}")
print(f"  Number of generalized velocities: {num_v}")
print(f"  Number of bodies: {plant.num_bodies()}")
print(f"  Number of actuators: {plant.num_actuators()}")

# Get joint information
print(f"\nJoints in the robot:")
for joint_idx in range(plant.num_joints()):
    joint = plant.get_joint(joint_idx)
    print(f"  {joint_idx}: {joint.name()} ({type(joint).__name__})")

## 3. Set Up MeshCat Visualizer

Initialize the MeshCat visualizer instance and configure it to connect with the Drake simulator for real-time visualization.

In [ ]:
# Start MeshCat
meshcat = StartMeshcat()

print(f"MeshCat is running at: http://localhost:7000")
print("Open this URL in your browser to view the visualization.")

## 4. Display Robot in MeshCat

Connect the Drake plant to the MeshCat visualizer to display the robot model with all its links and geometries.

In [ ]:
# Create a diagram and add the plant
builder = DiagramBuilder()
plant_copy = AddMultibodyPlantSceneGraph(builder, plant.time_step())

# Parse the URDF again for the copy
parser_copy = Parser(plant_copy.plant)
parser_copy.AddModelFromFile(robot_urdf_path)
plant_copy.plant.Finalize()

# Add visualizer
meshcat_params = MeshcatVisualizerParams(delete_on_initialization_event=False)
visualizer = builder.AddSystem(
    MeshcatVisualizer(meshcat, params=meshcat_params)
)

# Connect geometry query output to visualizer
builder.Connect(
    plant_copy.scene_graph.get_query_output_port(),
    visualizer.GetInputPort("geometry_query")
)

# Build the diagram
diagram = builder.Build()

# Create a simulator
simulator = Simulator(diagram)
context = simulator.get_mutable_context()

print("Robot successfully displayed in MeshCat!")

## 5. Configure Joint Controls

Set up joint actuators and define the control interface for manipulating individual robot joints.

In [ ]:
# Get joint information and create a mapping
joint_limits = {}
joint_names = []

for joint_idx in range(plant.num_joints()):
    joint = plant.get_joint(joint_idx)
    joint_names.append(joint.name())
    
    # Get joint limits
    if hasattr(joint, 'position_lower_limits'):
        lower = joint.position_lower_limits()
        upper = joint.position_upper_limits()
        if len(lower) > 0 and len(upper) > 0:
            joint_limits[joint.name()] = (lower[0], upper[0])
        else:
            # Default limits if not specified
            joint_limits[joint.name()] = (-np.pi, np.pi)
    else:
        joint_limits[joint.name()] = (-np.pi, np.pi)

print("Joint Control Configuration:")
for joint_name, (lower, upper) in joint_limits.items():
    print(f"  {joint_name}: [{lower:.3f}, {upper:.3f}]")

## 6. Animate Robot Movements

Create trajectories and animations to demonstrate robot movements over time using Drake's trajectory optimization.

In [ ]:
def animate_robot(joint_trajectory_func, duration=5.0, num_steps=50):
    """
    Animate the robot using a trajectory function.
    
    Parameters:
    - joint_trajectory_func: Function that takes time (0 to 1) and returns joint positions
    - duration: Duration of animation in seconds
    - num_steps: Number of animation frames
    """
    context = diagram.CreateDefaultContext()
    plant_context = plant_copy.plant.GetMyContextFromRoot(context)
    
    for i in range(num_steps):
        t = i / num_steps  # Normalized time [0, 1]
        joint_positions = joint_trajectory_func(t)
        plant_copy.plant.SetPositions(plant_context, joint_positions)
        simulator.Publish(context)
    
    print(f"Animation completed: {num_steps} frames over {duration} seconds")

# Example: Create a simple trajectory that moves the first joint
def simple_trajectory(t):
    """Simple trajectory that oscillates the first joint"""
    q = plant.GetPositions(simulator.get_context())
    if len(q) > 0:
        q[0] = np.sin(2 * np.pi * t) * 0.5  # Oscillate first joint
    return q

print("Animation function defined. Use animate_robot(trajectory_func) to run animations.")

## 7. Interactive Joint Manipulation

Implement interactive sliders or controls to allow real-time manipulation of joint angles and visualization of resulting poses in MeshCat.

In [ ]:
# Create interactive sliders for each joint
sliders = {}
slider_layout = []

for i, joint_name in enumerate(joint_names):
    if i < num_q:  # Only create sliders for configurable joints
        lower, upper = joint_limits[joint_name]
        slider = FloatSlider(
            value=0.0,
            min=lower,
            max=upper,
            step=0.01,
            description=f"Joint {i}: {joint_name}",
            orientation='horizontal',
            continuous_update=True
        )
        sliders[joint_name] = slider
        slider_layout.append(slider)

def update_robot_pose(*args):
    """Update the robot pose based on slider values"""
    context = diagram.CreateDefaultContext()
    plant_context = plant_copy.plant.GetMyContextFromRoot(context)
    
    # Create position vector from slider values
    q = np.zeros(num_q)
    for i, joint_name in enumerate(joint_names):
        if i < num_q and joint_name in sliders:
            q[i] = sliders[joint_name].value
    
    # Set plant positions
    plant_copy.plant.SetPositions(plant_context, q)
    
    # Update visualization
    simulator.Publish(context)

# Connect slider updates to robot pose update
for slider in sliders.values():
    slider.observe(update_robot_pose, names='value')

# Display controls
print(f"Interactive Joint Controls ({len(sliders)} joints)")
print("Adjust sliders to manipulate the robot in real-time.\n")
control_panel = VBox(slider_layout)
display(control_panel)

# Initialize the robot display
update_robot_pose()

## Additional Utilities

Helper functions for advanced manipulation and visualization.

In [ ]:
def get_current_pose():
    """Get current robot pose from sliders"""
    q = np.zeros(num_q)
    for i, joint_name in enumerate(joint_names):
        if i < num_q and joint_name in sliders:
            q[i] = sliders[joint_name].value
    return q

def set_pose(q):
    """Set robot pose programmatically"""
    for i, joint_name in enumerate(joint_names):
        if i < len(q) and joint_name in sliders:
            sliders[joint_name].value = q[i]

def reset_pose():
    """Reset all joints to zero"""
    for slider in sliders.values():
        slider.value = 0.0

def print_pose():
    """Print current pose to console"""
    q = get_current_pose()
    print("Current Joint Positions:")
    for i, joint_name in enumerate(joint_names):
        if i < num_q:
            print(f"  {joint_name}: {q[i]:.4f}")

# Example usage:
print("Available utility functions:")
print("  - get_current_pose(): Returns current joint positions")
print("  - set_pose(q): Set robot to configuration q")
print("  - reset_pose(): Reset all joints to zero")
print("  - print_pose(): Print current joint positions")